In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
metadata_df = (
    spark.table("workspace.metadata.metadata_config")
    .filter(col("active_flag")== "Y")
)

In [0]:
for row in metadata_df.collect():

    table_name = row["table_name"]
    source_table = row["target_table"]      # Bronze table
    primary_key = row["primary_key"]
    watermark_column = row["watermark_column"]
    scd_type = row["scd_type"]

    # Create Silver table name dynamically
    silver_table = source_table.replace(".bronze.", ".silver.")

    print("=" * 60)
    print(f"Processing Table : {table_name}")
    print(f"Source Table     : {source_table}")
    print(f"Target Table     : {silver_table}")
    print(f"Primary Key      : {primary_key}")
    print(f"Watermark Column : {watermark_column}")
    print(f"scd Type        : {scd_type}")

Processing Table : users
Source Table     : workspace.bronze.users
Target Table     : workspace.silver.users
Primary Key      : user_id
Watermark Column : created_at
scd Type        : SCD2
Processing Table : hosts
Source Table     : workspace.bronze.hosts
Target Table     : workspace.silver.hosts
Primary Key      : host_id
Watermark Column : joined_at
scd Type        : SCD2
Processing Table : properties
Source Table     : workspace.bronze.properties
Target Table     : workspace.silver.properties
Primary Key      : property_id
Watermark Column : created_at
scd Type        : SCD1
Processing Table : bookings
Source Table     : workspace.bronze.bookings
Target Table     : workspace.silver.bookings
Primary Key      : booking_id
Watermark Column : updated_at
scd Type        : SCD1
Processing Table : payments
Source Table     : workspace.bronze.payments
Target Table     : workspace.silver.payments
Primary Key      : payment_id
Watermark Column : payment_date
scd Type        : SCD1
Processing 

In [0]:
def apply_transformations(df, table_name):

    # Remove exact duplicate rows
    df = df.dropDuplicates()

    # Trim all string columns
    string_cols = [
        field.name
        for field in df.schema.fields
        if field.dataType.simpleString() == "string"
    ]

    for col_name in string_cols:
        df = df.withColumn(col_name, trim(col(col_name)))

    # -----------------------------
    # Users
    # -----------------------------
    if table_name == "users":

        if "email" in df.columns:
            df = df.withColumn("email", lower(col("email")))

        if "country" in df.columns:
            df = df.withColumn("country", upper(col("country")))

        if "user_type" in df.columns:
            df = df.withColumn("user_type", upper(col("user_type")))

    # -----------------------------
    # Hosts
    # -----------------------------
    elif table_name == "hosts":

        if "email" in df.columns:
            df = df.withColumn("email", lower(col("email")))

        if "country" in df.columns:
            df = df.withColumn("country", upper(col("country")))

    # -----------------------------
    # Properties
    # -----------------------------
    elif table_name == "properties":

        if "property_type" in df.columns:
            df = df.withColumn("property_type", upper(col("property_type")))

    # -----------------------------
    # Bookings
    # -----------------------------
    elif table_name == "bookings":

        if "status" in df.columns:
            df = df.withColumn("status", upper(col("status")))

    # -----------------------------
    # Payments
    # -----------------------------
    elif table_name == "payments":

        if "payment_method" in df.columns:
            df = df.withColumn("payment_method", upper(col("payment_method")))

        if "status" in df.columns:
            df = df.withColumn("status", upper(col("status")))

    # -----------------------------
    # Booking Updates
    # -----------------------------
    elif table_name == "booking_updates":

        if "status" in df.columns:
            df = df.withColumn("status", upper(col("status")))

    return df

In [0]:
def apply_data_quality(df, table_name, primary_key):

    # -----------------------------
    # Primary Key Validation
    # -----------------------------
    valid_df = df.filter(col(primary_key).isNotNull())
    invalid_df = df.filter(col(primary_key).isNull())

    # Remove duplicate primary keys
    valid_df = valid_df.dropDuplicates([primary_key])

    # -----------------------------
    # Users
    # -----------------------------
    if table_name == "users":

        valid_df = valid_df.filter(
            col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
        )

    # -----------------------------
    # Hosts
    # -----------------------------
    elif table_name == "hosts":

        valid_df = valid_df.filter(
            col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
        )

        valid_df = valid_df.filter(col("rating").between(0, 5))

    # -----------------------------
    # Properties
    # -----------------------------
    elif table_name == "properties":

        valid_df = valid_df.filter(col("base_price") > 0)
        valid_df = valid_df.filter(col("max_guests") > 0)

    # -----------------------------
    # Bookings
    # -----------------------------
    elif table_name == "bookings":

        valid_df = valid_df.filter(col("total_amount") > 0)
        valid_df = valid_df.filter(col("guests_count") > 0)
        valid_df = valid_df.filter(col("check_in") < col("check_out"))

    # -----------------------------
    # Payments
    # -----------------------------
    elif table_name == "payments":

        valid_df = valid_df.filter(col("amount") > 0)

    # -----------------------------
    # Booking Updates
    # -----------------------------
    elif table_name == "booking_updates":

        valid_df = valid_df.filter(col("total_amount") > 0)
        valid_df = valid_df.filter(col("guests_count") > 0)
        valid_df = valid_df.filter(col("check_in") < col("check_out"))

    return valid_df, invalid_df

In [0]:
from pyspark.sql.functions import current_timestamp, lit

def add_audit_columns(df, batch_id, source_file):

    df = (
        df.withColumn("LoadTimestamp", current_timestamp())
          .withColumn("BatchID", lit(batch_id))
          .withColumn("SourceFile", lit(source_file))
    )

    return df

In [0]:
from delta.tables import DeltaTable

def merge_scd_type1(valid_df, silver_table, primary_key):

    # -----------------------------
    # First Load
    # -----------------------------
    if not spark.catalog.tableExists(silver_table):

        (
            valid_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table)
        )

        print(f"Created Silver table : {silver_table}")

    # -----------------------------
    # Incremental Load (SCD Type 1)
    # -----------------------------
    else:

        delta_table = DeltaTable.forName(spark, silver_table)

        # Do not update the Primary Key
        update_set = {
            col: f"source.{col}"
            for col in valid_df.columns
            if col != primary_key
        }

        # Insert all columns
        insert_values = {
            col: f"source.{col}"
            for col in valid_df.columns
        }

        (
            delta_table.alias("target")
            .merge(
                valid_df.alias("source"),
                f"target.{primary_key} = source.{primary_key}"
            )
            .whenMatchedUpdate(
                set=update_set
            )
            .whenNotMatchedInsert(
                values=insert_values
            )
            .execute()
        )

        print(f"Merged into Silver table : {silver_table}")

In [0]:
from pyspark.sql.functions import sha2, concat_ws, col

def add_record_hash(df, table_name):

    hash_columns = {
        "users": [
            "email",
            "name",
            "country",
            "user_type",
            "is_business",
            "company_name"
        ],
        "hosts": [
            "name",
            "email",
            "phone",
            "is_verified",
            "is_active",
            "rating",
            "country"
        ]
    }

    columns = hash_columns.get(table_name)

    if columns is None:
        return df

    return df.withColumn(
        "RecordHash",
        sha2(concat_ws("||", *[col(c).cast("string") for c in columns]), 256)
    )

SCD type 2 implementation


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    current_timestamp,
    lit,
    col
)

def merge_scd_type2(valid_df, silver_table, primary_key):

    # ----------------------------------------
    # First Load
    # ----------------------------------------
    if not spark.catalog.tableExists(silver_table):

        (
            valid_df
            .withColumn("EffectiveStartDate", current_timestamp())
            .withColumn("EffectiveEndDate", lit(None).cast("timestamp"))
            .withColumn("IsCurrent", lit(True))
            .write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table)
        )

        print(f"Created Silver table : {silver_table}")

    # ----------------------------------------
    # Incremental Load
    # ----------------------------------------
    else:

        delta_table = DeltaTable.forName(spark, silver_table)

        # Current active records
        current_df = (
            spark.table(silver_table)
            .filter(col("IsCurrent") == True)
            .select(primary_key, "RecordHash")
        )

        # Records whose business data changed
        changed_df = (
            valid_df.alias("source")
            .join(current_df.alias("target"), primary_key)
            .filter(col("source.RecordHash") != col("target.RecordHash"))
        )

        # ----------------------------------------
        # Expire Old Version
        # ----------------------------------------
        (
            delta_table.alias("target")
            .merge(
                changed_df.alias("source"),
                f"""
                target.{primary_key}=source.{primary_key}
                AND target.IsCurrent = true
                """
            )
            .whenMatchedUpdate(
                set={
                    "IsCurrent": "false",
                    "EffectiveEndDate": "current_timestamp()"
                }
            )
            .execute()
        )

        # ----------------------------------------
        # Insert New Version
        # ----------------------------------------
        insert_df = (
            valid_df.alias("source")
            .join(
                spark.table(silver_table)
                .filter(col("IsCurrent") == True)
                .select(primary_key),
                primary_key,
                "left_anti"
            )
            .withColumn("EffectiveStartDate", current_timestamp())
            .withColumn("EffectiveEndDate", lit(None).cast("timestamp"))
            .withColumn("IsCurrent", lit(True))
        )

        (
            insert_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(silver_table)
        )

        print(f"Merged into Silver table : {silver_table}")

In [0]:
from pyspark.sql.types import *

audit_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("rows_read", LongType(), True),
    StructField("rows_written", LongType(), True),
    StructField("rows_rejected", LongType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("duration_seconds", DoubleType(), True)
])


def log_audit(
    run_id,
    table_name,
    layer,
    load_type,
    rows_read,
    rows_written,
    rows_rejected,
    status,
    error_message,
    start_time,
    end_time,
    duration_seconds
):

    audit_data = [(
        str(run_id),
        str(table_name),
        str(layer),
        str(load_type),
        int(rows_read),
        int(rows_written),
        int(rows_rejected),
        str(status),
        error_message if error_message is not None else "",
        start_time,
        end_time,
        float(duration_seconds)
    )]

    audit_df = spark.createDataFrame(
        audit_data,
        schema=audit_schema
    )

    (
        audit_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("workspace.metadata.audit_log")
    )

In [0]:
# =====================================================
# Process All Tables from Metadata
# =====================================================

for row in metadata_df.collect():

    table_name = row["table_name"]
    source_table = row["target_table"]
    primary_key = row["primary_key"]
    watermark_column = row["watermark_column"]
    load_type = row["load_type"]
    scd_type = row["scd_type"]

    silver_table = source_table.replace(".bronze.", ".silver.")

    # Audit Information
    batch_id = str(uuid.uuid4())
    source_file = source_table

    print(f"\nProcessing Table : {table_name}")

    try:

        # -----------------------------------------
        # Start Timer
        # -----------------------------------------
        start_time = datetime.now()

        # -----------------------------------------
        # Read Bronze Table
        # -----------------------------------------
        bronze_df = spark.table(source_table)

        rows_read = bronze_df.count()

        # -----------------------------------------
        # Apply Transformations
        # -----------------------------------------
        transformed_df = apply_transformations(
            bronze_df,
            table_name
        )

        # -----------------------------------------
        # Data Quality
        # -----------------------------------------
        valid_df, invalid_df = apply_data_quality(
            transformed_df,
            table_name,
            primary_key
        )

        # -----------------------------------------
        # Add Audit Columns
        # -----------------------------------------
        valid_df = add_audit_columns(
            valid_df,
            batch_id,
            source_file
        )

        # -----------------------------------------
        # Apply SCD
        # -----------------------------------------
        if scd_type == "SCD1":

            merge_scd_type1(
                valid_df,
                silver_table,
                primary_key
            )

        elif scd_type == "SCD2":

            valid_df = add_record_hash(
                valid_df,
                table_name
            )

            merge_scd_type2(
                valid_df,
                silver_table,
                primary_key
            )

        # -----------------------------------------
        # Metrics
        # -----------------------------------------
        rows_written = valid_df.count()
        rows_rejected = invalid_df.count()

        end_time = datetime.now()
        duration_seconds = (
            end_time - start_time
        ).total_seconds()

        print(f" {table_name} processed successfully.")
        print(f"Rows Read      : {rows_read}")
        print(f"Rows Written   : {rows_written}")
        print(f"Rows Rejected  : {rows_rejected}")

        # -----------------------------------------
        # Audit Logging
        # -----------------------------------------
        log_audit(
            run_id=batch_id,
            table_name=table_name,
            layer="Silver",
            load_type=load_type,
            rows_read=rows_read,
            rows_written=rows_written,
            rows_rejected=rows_rejected,
            status="SUCCESS",
            error_message=None,
            start_time=start_time,
            end_time=end_time,
            duration_seconds=duration_seconds
        )

    except Exception as e:

        end_time = datetime.now()
        duration_seconds = (
            end_time - start_time
        ).total_seconds()

        print(f" Error processing {table_name}")
        print(str(e))

        log_audit(
            run_id=batch_id,
            table_name=table_name,
            layer="Silver",
            load_type=load_type,
            rows_read=rows_read if "rows_read" in locals() else 0,
            rows_written=0,
            rows_rejected=0,
            status="FAILED",
            error_message=str(e),
            start_time=start_time,
            end_time=end_time,
            duration_seconds=duration_seconds
        )


Processing Table : users
Merged into Silver table : workspace.silver.users
 users processed successfully.
Rows Read      : 124509
Rows Written   : 124259
Rows Rejected  : 0

Processing Table : hosts
Merged into Silver table : workspace.silver.hosts
 hosts processed successfully.
Rows Read      : 38768
Rows Written   : 19384
Rows Rejected  : 0

Processing Table : properties
Merged into Silver table : workspace.silver.properties
 properties processed successfully.
Rows Read      : 36326
Rows Written   : 18163
Rows Rejected  : 0

Processing Table : bookings
Merged into Silver table : workspace.silver.bookings
 bookings processed successfully.
Rows Read      : 144494
Rows Written   : 72246
Rows Rejected  : 0

Processing Table : payments
Merged into Silver table : workspace.silver.payments
 payments processed successfully.
Rows Read      : 99276
Rows Written   : 41592
Rows Rejected  : 0

Processing Table : booking_updates
Merged into Silver table : workspace.silver.booking_updates
 booking

In [0]:
spark.table("workspace.metadata.audit_log").printSchema()

root
 |-- run_id: string (nullable = true)
 |-- table_name: string (nullable = true)
 |-- layer: string (nullable = true)
 |-- load_type: string (nullable = true)
 |-- rows_read: long (nullable = true)
 |-- rows_written: long (nullable = true)
 |-- rows_rejected: long (nullable = true)
 |-- status: string (nullable = true)
 |-- error_message: string (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- duration_seconds: double (nullable = true)



In [0]:
display(
    spark.table("workspace.metadata.audit_log")
    .orderBy(col("start_time").desc())
)

run_id,table_name,layer,load_type,rows_read,rows_written,rows_rejected,status,error_message,start_time,end_time,duration_seconds
14ce67c2-107d-41ff-8e8f-f68b53d7e3f2,booking_updates,Silver,INCREMENTAL,166136,75112,0,SUCCESS,,2026-07-28T10:47:29.817Z,2026-07-28T10:47:36.103Z,6.286359
3b7b1756-5a73-4719-af16-9aa8e002a810,payments,Silver,INCREMENTAL,99276,41592,0,SUCCESS,,2026-07-28T10:47:22.040Z,2026-07-28T10:47:28.379Z,6.339033
44dcbc00-8e5f-4d3d-bc7f-2444c78f0855,bookings,Silver,INCREMENTAL,144494,72246,0,SUCCESS,,2026-07-28T10:47:14.269Z,2026-07-28T10:47:20.607Z,6.338257
c0651665-2edc-4051-90ee-5ffbbcd7916b,properties,Silver,INCREMENTAL,36326,18163,0,SUCCESS,,2026-07-28T10:47:05.802Z,2026-07-28T10:47:12.567Z,6.76442
1bb65711-247d-416e-97e0-aabea16adae1,hosts,Silver,INCREMENTAL,38768,19384,0,SUCCESS,,2026-07-28T10:46:57.349Z,2026-07-28T10:47:04.208Z,6.859325
c2e2d595-ad9c-410f-9d2c-f0d71f536ef3,users,Silver,INCREMENTAL,124509,124259,0,SUCCESS,,2026-07-28T10:46:49.192Z,2026-07-28T10:46:55.884Z,6.691718
088d1d1f-4d47-4c49-a1f7-438805a93d81,booking_updates,Silver,INCREMENTAL,166136,0,0,FAILED,[CANNOT_DETERMINE_TYPE] Some of types cannot be determined after inferring.,2026-07-28T10:44:22.719Z,2026-07-28T10:44:28.969Z,6.249349
112ac614-a4d7-486a-856d-5ed58a539497,payments,Silver,INCREMENTAL,99276,0,0,FAILED,[CANNOT_DETERMINE_TYPE] Some of types cannot be determined after inferring.,2026-07-28T10:44:15.312Z,2026-07-28T10:44:21.302Z,5.989773
5768b6f7-bbfd-4a8a-b555-a45b3f98d3b2,bookings,Silver,INCREMENTAL,144494,0,0,FAILED,[CANNOT_DETERMINE_TYPE] Some of types cannot be determined after inferring.,2026-07-28T10:44:07.659Z,2026-07-28T10:44:13.925Z,6.266121
8a220ac1-f797-4787-a5f1-20186cc6de7a,properties,Silver,INCREMENTAL,36326,0,0,FAILED,[CANNOT_DETERMINE_TYPE] Some of types cannot be determined after inferring.,2026-07-28T10:43:59.643Z,2026-07-28T10:44:06.152Z,6.509818
